In [2]:
# NOTEBOOK 16: CWQ FROZEN TEST AND COST-MATCHED EVALUATION
# Run this entire cell in a NEW Colab notebook with a T4 GPU runtime.

import importlib.metadata as metadata
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
summary_path = (
    ROOT
    / "outputs/cwq_controller_training_v1"
    / "cwq_controller_training_summary_v1.json"
)

if not summary_path.exists():
    raise FileNotFoundError(
        f"Missing CWQ controller report: {summary_path}"
    )

versions = json.loads(
    summary_path.read_text(encoding="utf-8")
)["dependency_versions"]

packages = [
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "safetensors",
]

needs_install = False

for package in packages:
    try:
        needs_install |= metadata.version(package) != versions[package]
    except metadata.PackageNotFoundError:
        needs_install = True

if needs_install:
    torch_version = metadata.version("torch").split("+")[0]

    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".txt"
    ) as constraints:
        constraints.write(f"torch=={torch_version}\n")
        constraints.flush()

        result = subprocess.run([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-c",
            constraints.name,
            *[f"{p}=={versions[p]}" for p in packages],
        ])

        if result.returncode:
            raise RuntimeError(
                "Dependency installation failed. "
                "Read the pip error above."
            )

# A fresh Python process prevents stale imports after installation.
# The program is passed in memory; no separate code file is created.
import collections
import csv
import hashlib
import importlib.metadata
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")

SEED = 42
CONTROLLER_BATCH = 64
ANSWER_BATCH = 128
BOOTSTRAP_SAMPLES = 5000
DEV_REGRESSION_QUESTIONS = 3
CODE_VERSION = "cwq_frozen_test_v1_0"

GAIN_WEIGHTS = np.asarray(
    [0.0, 1.0, 2.0, 3.5], dtype=np.float64
)

METHODS = [
    "relation_fixed_b14_e2e",
    "relation_fixed_e2e",
    "model_fixed_b14_e2e",
    "model_fixed_e2e",
    "setstop_e2e",
]

METHOD_NAMES = {
    "relation_fixed_b14_e2e": "Relation fixed, cap 14",
    "relation_fixed_e2e": "Relation fixed, cap 24",
    "model_fixed_b14_e2e": "Learned ranker, cap 14",
    "model_fixed_e2e": "Learned ranker, cap 24",
    "setstop_e2e": "SetStop, maximum cap 24",
}


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, obj):
    temporary = path.with_name(path.name + ".partial")

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            obj,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    temporary.replace(path)


def file_hash(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(8 * 1024**2), b""
        ):
            digest.update(block)

    return digest.hexdigest()


def fingerprint(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True).encode()
    ).hexdigest()


def verify_inventory(path, inventory):
    entry = next(
        (
            item
            for item in inventory["files"]
            if item["name"] == path.name
        ),
        None,
    )

    if (
        entry is None
        or not path.exists()
        or path.stat().st_size != entry["bytes"]
        or file_hash(path) != entry["sha256"]
    ):
        raise ValueError(
            f"Inventory verification failed: {path}"
        )


def tokens(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " "),
        )
    )


def parse_question(
    row, entity_to_id, relation_to_id, relation_names
):
    def entity(value):
        if isinstance(value, dict):
            value = value.get(
                "kb_id",
                value.get("id", value.get("entity")),
            )

        if isinstance(value, int) and not isinstance(value, bool):
            result = (
                value
                if 0 <= value < len(entity_to_id)
                else None
            )
        elif isinstance(value, str):
            result = entity_to_id.get(value)
        else:
            result = None

        if result is None:
            raise ValueError(f"Unmapped entity: {value}")

        return result

    def relation(value):
        if isinstance(value, int) and not isinstance(value, bool):
            result = (
                value
                if 0 <= value < len(relation_names)
                else None
            )
        elif isinstance(value, str):
            result = relation_to_id.get(value)
        else:
            result = None

        if result is None:
            raise ValueError(f"Unmapped relation: {value}")

        return result

    qid = str(row["id"])
    question = row["question"].strip()
    subgraph = row["subgraph"]

    raw_edges = subgraph.get(
        "tuples", subgraph.get("triples")
    )

    if not isinstance(raw_edges, list):
        raise ValueError(f"Missing graph triples: {qid}")

    nodes = {
        entity(value)
        for value in subgraph["entities"]
    }

    edges = set()

    for edge in raw_edges:
        if (
            not isinstance(edge, (list, tuple))
            or len(edge) != 3
        ):
            raise ValueError(f"Invalid graph edge: {qid}")

        edges.add((
            entity(edge[0]),
            relation(edge[1]),
            entity(edge[2]),
        ))

    nodes |= {
        node
        for source, rel, target in edges
        for node in (source, target)
    }

    topics = {
        entity(value)
        for value in row.get(
            "entities", row.get("topic_entities", [])
        )
    }

    gold = {
        entity(value)
        for value in row["answers"]
    }

    if not (
        topics
        and topics <= nodes
        and gold <= nodes
        and 2 <= len(gold) <= 50
    ):
        raise ValueError(
            f"Question no longer satisfies CWQ eligibility: {qid}"
        )

    grouped = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for source, rel, target in sorted(edges):
        grouped[source][rel].add(target)

    adjacency = {
        source: {
            rel: sorted(targets)
            for rel, targets in group.items()
        }
        for source, group in grouped.items()
    }

    # Gold remains separate from the graph passed into search.
    graph = dict(
        qid=qid,
        question=question,
        topics=sorted(topics),
        adjacency=adjacency,
        input_edges=len(raw_edges),
        unique_edges=len(edges),
        graph_nodes=len(nodes),
    )

    return graph, gold


def frontier(
    graph, depth, expanded, relation_tokens, max_hops
):
    question_tokens = tokens(graph["question"])

    branches = [
        (source, rel, depth[source], targets)
        for source in sorted(depth)
        if depth[source] < max_hops
        for rel, targets in sorted(
            graph["adjacency"].get(source, {}).items()
        )
        if (source, rel) not in expanded
    ]

    return sorted(
        branches,
        key=lambda branch: (
            -len(
                question_tokens
                & relation_tokens[branch[1]]
            ),
            branch[2],
            -len(branch[3]),
            branch[1],
            branch[0],
        ),
    )


def controller_text(
    question,
    predicted_count,
    visited_count,
    step,
    branch,
    history,
    relations,
):
    history_text = (
        " -> ".join(history[-6:])
        if history
        else "none"
    )

    return (
        f"Question: {question}\n"
        f"Answers found so far: {predicted_count}\n"
        f"Visited entities: {visited_count}\n"
        f"Search step: {step}\n"
        f"Source depth: {branch[2]}\n"
        f"Candidate relation: {relations[branch[1]]}\n"
        f"Candidate target count: {len(branch[3])}\n"
        f"Recent relations: {history_text}"
    )


def answer_text(
    question,
    branch,
    source_path,
    target,
    adjacency,
    relations,
):
    path = source_path + [branch[1]]
    path_text = " -> ".join(
        relations[rel] for rel in path
    )

    return (
        f"Question: {question}\n"
        f"Candidate relation path: {path_text}\n"
        f"Candidate depth: {branch[2] + 1}\n"
        f"Incoming relation: {relations[branch[1]]}\n"
        f"Entities returned by final relation: {len(branch[3])}\n"
        f"Candidate outgoing relation count: "
        f"{len(adjacency.get(target, {}))}"
    )


def run_search(
    graph,
    relations,
    relation_tokens,
    controller_predict,
    answer_predict,
    max_hops,
    frontier_limit,
    action_cap,
    controller_threshold,
    answer_threshold,
    learned,
    matched_budget,
):
    depth = {
        node: 0
        for node in graph["topics"]
    }
    paths = {
        node: []
        for node in graph["topics"]
    }

    expanded = set()
    predicted = set()
    node_scores = {}
    history = []
    trace = []

    actions = 0
    edges = 0
    truncations = 0
    controller_candidates = 0
    answer_candidates = 0

    adaptive = None
    matched = None

    def snapshot(reason):
        return dict(
            actions=actions,
            retrieved_edges=edges,
            frontier_truncations=truncations,
            controller_candidates_scored=controller_candidates,
            answer_candidates_scored=answer_candidates,
            predicted_answers=sorted(predicted),
            visited_depths=[
                [node, node_depth]
                for node, node_depth in sorted(depth.items())
            ],
            scored_entities=sorted(node_scores),
            expanded_branches=[
                list(branch)
                for branch in sorted(expanded)
            ],
            stop_reason=reason,
            trace=[dict(item) for item in trace],
        )

    while actions < action_cap:
        full = frontier(
            graph,
            depth,
            expanded,
            relation_tokens,
            max_hops,
        )

        if not full:
            break

        pool = full[:frontier_limit]
        truncations += int(len(full) > len(pool))

        maximum_continue = None
        should_stop = False

        if learned:
            texts = [
                controller_text(
                    graph["question"],
                    len(predicted),
                    len(depth),
                    actions,
                    branch,
                    history,
                    relations,
                )
                for branch in pool
            ]

            probabilities = np.asarray(
                controller_predict(texts),
                dtype=np.float32,
            )

            if (
                probabilities.shape != (len(pool), 4)
                or not np.isfinite(probabilities).all()
            ):
                raise ValueError(
                    "Invalid controller output"
                )

            if (
                np.any(probabilities < 0)
                or not np.allclose(
                    probabilities.sum(axis=1),
                    1,
                    atol=1e-4,
                )
            ):
                raise ValueError(
                    "Controller probabilities do not sum to one"
                )

            controller_candidates += len(pool)

            maximum_continue = float(
                np.max(1.0 - probabilities[:, 0])
            )

            should_stop = (
                maximum_continue < controller_threshold
            )

            index = int(
                np.argmax(probabilities @ GAIN_WEIGHTS)
            )
        else:
            index = 0

        trace.append(dict(
            step=actions,
            visited_count=len(depth),
            predicted_answer_count=len(predicted),
            full_frontier_count=len(full),
            shortlist_count=len(pool),
            max_continue_score=maximum_continue,
            threshold_would_stop=bool(should_stop),
            next_source=pool[index][0],
            next_relation=relations[pool[index][1]],
        ))

        if learned and should_stop and adaptive is None:
            adaptive = snapshot("learned_stop")

        # Continue the fixed-budget trajectory.
        # SetStop is the prefix saved at its first stopping decision.
        source, rel, source_depth, targets = pool[index]
        source_path = list(paths[source])

        texts = [
            answer_text(
                graph["question"],
                pool[index],
                source_path,
                target,
                graph["adjacency"],
                relations,
            )
            for target in targets
        ]

        answer_probabilities = np.asarray(
            answer_predict(texts),
            dtype=np.float32,
        )

        if (
            answer_probabilities.shape != (len(targets),)
            or not np.isfinite(answer_probabilities).all()
        ):
            raise ValueError(
                "Invalid answer-recognizer output"
            )

        if np.any(
            (answer_probabilities < 0)
            | (answer_probabilities > 1)
        ):
            raise ValueError(
                "Answer probabilities outside [0, 1]"
            )

        expanded.add((source, rel))

        for target, probability in zip(
            targets, answer_probabilities
        ):
            new_depth = source_depth + 1

            if (
                target not in depth
                or new_depth < depth[target]
            ):
                depth[target] = new_depth
                paths[target] = source_path + [rel]

            best_score = max(
                node_scores.get(target, 0.0),
                float(probability),
            )
            node_scores[target] = best_score

            if best_score >= answer_threshold:
                predicted.add(target)

        actions += 1
        edges += len(targets)
        answer_candidates += len(targets)
        history.append(relations[rel])

        # Save cap-14 immediately after its final expansion,
        # before scoring any frontier for the next action.
        if actions == matched_budget:
            matched = snapshot("action_budget")

    fixed = snapshot(
        "action_budget"
        if actions >= action_cap
        else "empty_frontier"
    )

    if adaptive is None:
        adaptive = fixed

    if matched is None:
        matched = fixed

    return fixed, adaptive, matched


def remaining_reachable(graph, state, max_hops):
    # Post-search diagnostic; gold is not used here.
    expanded = {
        tuple(branch)
        for branch in state["expanded_branches"]
    }

    best = {}
    queue = collections.deque()

    for source, source_depth in state["visited_depths"]:
        if source_depth >= max_hops:
            continue

        for rel, targets in graph["adjacency"].get(
            source, {}
        ).items():
            if (source, rel) in expanded:
                continue

            for target in targets:
                candidate_depth = source_depth + 1

                if (
                    target not in best
                    or candidate_depth < best[target]
                ):
                    best[target] = candidate_depth
                    queue.append(target)

    while queue:
        node = queue.popleft()
        node_depth = best[node]

        if node_depth >= max_hops:
            continue

        for rel, targets in graph["adjacency"].get(
            node, {}
        ).items():
            if (node, rel) in expanded:
                continue

            for target in targets:
                candidate_depth = node_depth + 1

                if (
                    target not in best
                    or candidate_depth < best[target]
                ):
                    best[target] = candidate_depth
                    queue.append(target)

    return set(best)


def evaluate_state(
    graph, gold, state, policy, max_hops
):
    predicted = set(state["predicted_answers"])
    visited = {
        node
        for node, depth in state["visited_depths"]
    }
    scored = set(state["scored_entities"])

    if not (
        predicted <= visited
        and predicted <= scored
    ):
        raise ValueError(
            "A predicted answer was not visited/scored"
        )

    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    visited_gold = gold & visited

    reachable_missing = (
        (gold - visited)
        & remaining_reachable(graph, state, max_hops)
    )

    if fp == 0 and fn == 0:
        error = "exact"
    elif not predicted:
        error = "empty_prediction"
    elif fp == 0:
        error = "incomplete_only"
    elif fn == 0:
        error = "over_prediction_only"
    else:
        error = "mixed_fp_fn"

    return dict(
        qid=graph["qid"],
        policy=policy,
        precision=tp / max(1, len(predicted)),
        recall=tp / len(gold),
        answer_f1=2 * tp / (len(predicted) + len(gold)),
        exact_match=int(predicted == gold),
        complete_recall=int(gold <= predicted),
        true_positive_count=tp,
        false_positive_count=fp,
        false_negative_count=fn,
        gold_answer_count=len(gold),
        predicted_answer_count=len(predicted),
        predicted_answers=sorted(predicted),
        gold_answers=sorted(gold),
        actions=state["actions"],
        retrieved_edges=state["retrieved_edges"],
        input_edges=graph["input_edges"],
        fraction_of_input_graph_exposed=(
            state["retrieved_edges"]
            / max(1, graph["input_edges"])
        ),
        visited_entities=len(visited),
        visited_gold_count=len(visited_gold),
        visited_gold_recall=(
            len(visited_gold) / len(gold)
        ),
        missed_by_search_count=len(gold - visited),
        visited_gold_not_predicted_count=len(
            visited_gold - predicted
        ),
        recognizer_false_negative_count=len(
            (gold & scored) - predicted
        ),
        visited_gold_unscored_count=len(
            visited_gold - scored
        ),
        remaining_unvisited_reachable_gold_count=len(
            reachable_missing
        ),
        premature_stop=bool(
            state["stop_reason"] == "learned_stop"
            and reachable_missing
        ),
        incomplete_set=bool(fn),
        empty_prediction=not predicted,
        error_category=error,
        stop_reason=state["stop_reason"],
        frontier_truncations=state["frontier_truncations"],
        controller_candidates_scored=(
            state["controller_candidates_scored"]
        ),
        answer_candidates_scored=(
            state["answer_candidates_scored"]
        ),
    )


def evaluate_question(
    graph,
    gold,
    relations,
    relation_tokens,
    controller_predict,
    answer_predict,
    settings,
):
    kwargs = dict(
        max_hops=settings["max_hops"],
        frontier_limit=settings["frontier_limit"],
        action_cap=settings["action_cap"],
        controller_threshold=settings["controller_threshold"],
        answer_threshold=settings["answer_threshold"],
        matched_budget=settings["matched_budget"],
    )

    fixed, adaptive, matched_model = run_search(
        graph,
        relations,
        relation_tokens,
        controller_predict,
        answer_predict,
        **kwargs,
        learned=True,
    )

    relation, _, matched_relation = run_search(
        graph,
        relations,
        relation_tokens,
        controller_predict,
        answer_predict,
        **kwargs,
        learned=False,
    )

    for small, large in [
        (adaptive, fixed),
        (matched_model, fixed),
        (matched_relation, relation),
    ]:
        if (
            small["actions"] > large["actions"]
            or small["retrieved_edges"] > large["retrieved_edges"]
            or not (
                set(small["predicted_answers"])
                <= set(large["predicted_answers"])
            )
        ):
            raise ValueError(
                "Shared-prefix consistency check failed"
            )

    states = {
        "relation_fixed_b14_e2e": matched_relation,
        "relation_fixed_e2e": relation,
        "model_fixed_b14_e2e": matched_model,
        "model_fixed_e2e": fixed,
        "setstop_e2e": adaptive,
    }

    records = []

    for policy in METHODS:
        row = evaluate_state(
            graph,
            gold,
            states[policy],
            policy,
            settings["max_hops"],
        )

        row["budget_cap"] = (
            settings["matched_budget"]
            if "b14" in policy
            else settings["action_cap"]
        )

        if (
            row["actions"] > row["budget_cap"]
            or row["retrieved_edges"] > graph["unique_edges"]
        ):
            raise ValueError(
                "Action or edge count violates the protocol"
            )

        records.append(row)

    return dict(
        qid=graph["qid"],
        question=graph["question"],
        records=records,
        traces={
            policy: states[policy]["trace"]
            for policy in METHODS
        },
        shared_prefix_check=True,
    )


def summarize(records):
    summaries = {}

    for policy in METHODS:
        group = [
            row
            for row in records
            if row["policy"] == policy
        ]

        n = len(group)

        if not n:
            raise ValueError(
                f"No results for {policy}"
            )

        tp, fp, fn = (
            sum(row[key] for row in group)
            for key in [
                "true_positive_count",
                "false_positive_count",
                "false_negative_count",
            ]
        )

        def mean(key):
            return float(
                np.mean([row[key] for row in group])
            )

        result = dict(
            method=METHOD_NAMES[policy],
            questions=n,
            macro_precision=mean("precision"),
            macro_recall=mean("recall"),
            macro_f1=mean("answer_f1"),
            micro_f1=(
                2 * tp / max(1, 2 * tp + fp + fn)
            ),
            exact_match=mean("exact_match"),
            complete_recall=mean("complete_recall"),
            mean_actions=mean("actions"),
            mean_retrieved_edges=mean("retrieved_edges"),
            mean_fraction_of_input_graph_exposed=mean(
                "fraction_of_input_graph_exposed"
            ),
            visited_gold_recall=mean("visited_gold_recall"),
            empty_prediction_rate=mean("empty_prediction"),
            incomplete_set_rate=mean("incomplete_set"),
            premature_stop_fraction_all_questions=mean(
                "premature_stop"
            ),
            true_positive_count=tp,
            false_positive_count=fp,
            false_negative_count=fn,
            stop_reasons=dict(
                collections.Counter(
                    row["stop_reason"]
                    for row in group
                )
            ),
        )

        learned_stops = [
            row
            for row in group
            if row["stop_reason"] == "learned_stop"
        ]

        result["premature_stop_rate_given_learned_stop"] = (
            sum(
                row["premature_stop"]
                for row in learned_stops
            )
            / len(learned_stops)
            if learned_stops
            else None
        )

        summaries[policy] = result

    return summaries


def bootstrap_comparisons(records, qids):
    groups = {
        policy: {
            row["qid"]: row
            for row in records
            if row["policy"] == policy
        }
        for policy in METHODS
    }

    if any(
        set(group) != set(qids)
        for group in groups.values()
    ):
        raise ValueError(
            "Paired-bootstrap question sets differ"
        )

    metric_keys = {
        "macro_f1": "answer_f1",
        "macro_precision": "precision",
        "macro_recall": "recall",
        "exact_match": "exact_match",
        "complete_recall": "complete_recall",
        "mean_actions": "actions",
        "mean_retrieved_edges": "retrieved_edges",
    }

    results = {}
    n = len(qids)

    for baseline in METHODS[:-1]:
        difference = np.asarray(
            [
                [
                    groups["setstop_e2e"][qid][key]
                    - groups[baseline][qid][key]
                    for key in metric_keys.values()
                ]
                for qid in qids
            ],
            dtype=np.float64,
        )

        rng = np.random.default_rng(SEED)

        draws = np.empty(
            (
                BOOTSTRAP_SAMPLES,
                len(metric_keys),
            ),
            dtype=np.float64,
        )

        for start in range(
            0, BOOTSTRAP_SAMPLES, 250
        ):
            count = min(
                250, BOOTSTRAP_SAMPLES - start
            )

            indices = rng.integers(
                0, n, size=(count, n)
            )

            draws[start:start + count] = (
                difference[indices].mean(axis=1)
            )

        estimates = difference.mean(axis=0)

        low, high = np.quantile(
            draws, [0.025, 0.975], axis=0
        )

        results[baseline] = {
            name: dict(
                difference=float(estimates[index]),
                ci95=[
                    float(low[index]),
                    float(high[index]),
                ],
            )
            for index, name in enumerate(metric_keys)
        }

    return dict(
        comparison="SetStop minus comparator",
        questions=n,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        seed=SEED,
        resampling_unit="question",
        interval="paired percentile bootstrap, 95 percent",
        uncertainty_scope=(
            "Test-question sampling, conditional "
            "on the trained models."
        ),
        primary_comparator="model_fixed_b14_e2e",
        comparisons=results,
    )


def main():
    os.environ["USE_TF"] = "0"
    os.environ["USE_FLAX"] = "0"
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    import torch
    import transformers
    import peft

    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        BitsAndBytesConfig,
    )
    from peft import (
        PeftModel,
        prepare_model_for_kbit_training,
    )
    from safetensors import safe_open

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Select a T4 GPU runtime before running this cell."
        )

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    data = ROOT / "data/extracted/cwq"
    pre = ROOT / "outputs/cwq_preprocessing_v1"
    controller_output = (
        ROOT / "outputs/cwq_controller_training_v1"
    )
    answer_output = (
        ROOT / "outputs/cwq_answer_recognizer_v1"
    )
    dev_output = ROOT / "outputs/cwq_dev_online_v1"
    output = ROOT / "outputs/cwq_test_evaluation_v1"

    output.mkdir(parents=True, exist_ok=True)

    generation = read_json(
        pre / "cwq_state_generation_summary_v1.json"
    )
    pre_inventory = read_json(
        pre / "artifact_inventory_v1.json"
    )

    verify_inventory(
        pre / "cwq_state_generation_summary_v1.json",
        pre_inventory,
    )

    manifest_path = Path(
        generation["manifest_path"]
    )

    verify_inventory(
        manifest_path, pre_inventory
    )

    manifest = read_json(manifest_path)

    controller_inventory = read_json(
        controller_output / "artifact_inventory_v1.json"
    )
    answer_inventory = read_json(
        answer_output / "artifact_inventory_v1.json"
    )
    dev_inventory = read_json(
        dev_output / "artifact_inventory_v1.json"
    )

    for name in [
        "cwq_controller_training_summary_v1.json",
        "cwq_dev_stop_threshold_v1.json",
    ]:
        verify_inventory(
            controller_output / name,
            controller_inventory,
        )

    for name in [
        "cwq_answer_recognizer_summary_v1.json",
        "cwq_answer_recognizer_threshold_v1.json",
    ]:
        verify_inventory(
            answer_output / name,
            answer_inventory,
        )

    for name in [
        "cwq_dev_online_summary_v1.json",
        "cwq_dev_online_protocol_v1.json",
        "cwq_dev_online_per_question_v1.jsonl",
    ]:
        verify_inventory(
            dev_output / name,
            dev_inventory,
        )

    controller_summary = read_json(
        controller_output
        / "cwq_controller_training_summary_v1.json"
    )
    stop_report = read_json(
        controller_output
        / "cwq_dev_stop_threshold_v1.json"
    )
    answer_summary = read_json(
        answer_output
        / "cwq_answer_recognizer_summary_v1.json"
    )
    answer_report = read_json(
        answer_output
        / "cwq_answer_recognizer_threshold_v1.json"
    )
    dev_summary = read_json(
        dev_output / "cwq_dev_online_summary_v1.json"
    )
    dev_protocol = read_json(
        dev_output / "cwq_dev_online_protocol_v1.json"
    )

    signatures = [
        generation["signature"],
        manifest["signature"],
        pre_inventory["signature"],
        controller_summary["data_signature"],
        controller_inventory["data_signature"],
        answer_summary["data_signature"],
        answer_inventory["data_signature"],
        dev_summary["data_signature"],
        dev_inventory["data_signature"],
        dev_protocol["data_signature"],
    ]

    provenance_reports = [
        controller_summary,
        answer_summary,
        stop_report,
        answer_report,
        dev_summary,
        dev_protocol,
    ]

    if (
        len(set(signatures)) != 1
        or any(
            report.get("test_used", True)
            for report in provenance_reports
        )
    ):
        raise ValueError(
            "Data signatures or development-only "
            "provenance disagree"
        )

    if not (
        dev_summary["signature"]
        == dev_protocol["signature"]
        == dev_inventory["signature"]
    ):
        raise ValueError(
            "Notebook 15 report signatures disagree"
        )

    expected_versions = controller_summary[
        "dependency_versions"
    ]

    versions = {
        package: importlib.metadata.version(package)
        for package in [
            "transformers",
            "peft",
            "accelerate",
            "bitsandbytes",
            "safetensors",
        ]
    }

    if any(
        versions[package] != expected_versions[package]
        for package in versions
    ):
        raise ValueError(
            "Dependency versions differ from Notebook 13"
        )

    adapter_dir = Path(
        controller_summary["adapter_path"]
    )
    answer_dir = Path(
        answer_summary["saved_model"]
    )

    for name in [
        "controller_config.json",
        "adapter_config.json",
        "adapter_model.safetensors",
    ]:
        verify_inventory(
            adapter_dir / name,
            controller_inventory,
        )

    for name in [
        "answer_recognizer_config.json",
        "config.json",
        "model.safetensors",
    ]:
        verify_inventory(
            answer_dir / name,
            answer_inventory,
        )

    controller_config = read_json(
        adapter_dir / "controller_config.json"
    )
    answer_config = read_json(
        answer_dir / "answer_recognizer_config.json"
    )

    if controller_config["class_to_id"] != {
        "0": 0,
        "1": 1,
        "2": 2,
        "3_plus": 3,
    }:
        raise ValueError(
            "Unexpected controller class order"
        )

    if answer_config["class_to_id"] != {
        "not_answer": 0,
        "answer": 1,
    }:
        raise ValueError(
            "Unexpected answer class order"
        )

    dev_mean = float(
        dev_summary["methods"]["setstop_e2e"]["mean_actions"]
    )

    # Selected using development only.
    matched_budget = int(
        np.floor(dev_mean + 0.5)
    )

    settings = dict(
        max_hops=generation["configuration"]["max_hops"],
        frontier_limit=(
            generation["configuration"]["frontier_limit"]
        ),
        action_cap=generation["configuration"]["action_cap"],
        controller_threshold=float(
            stop_report["selected_threshold"]
        ),
        answer_threshold=float(
            answer_report["selected_threshold"]
        ),
        matched_budget=matched_budget,
    )

    expected_settings = dict(
        max_hops=4,
        frontier_limit=128,
        action_cap=24,
        controller_threshold=0.853,
        answer_threshold=0.96,
        matched_budget=14,
    )

    if (
        settings != expected_settings
        or {
            key: settings[key]
            for key in dev_summary["settings"]
        } != dev_summary["settings"]
    ):
        raise ValueError(
            "Frozen settings differ from the "
            "agreed Notebook 15 protocol"
        )

    if not (
        settings["max_hops"]
        == controller_config["max_hops"]
        == answer_config["max_hops"]
        and settings["frontier_limit"]
        == controller_config["frontier_limit"]
        and settings["action_cap"]
        == controller_config["max_actions"]
        and settings["controller_threshold"]
        == answer_report["controller_threshold_unchanged"]
    ):
        raise ValueError(
            "Model configurations disagree "
            "with evaluation settings"
        )

    selected = manifest["splits"]["test"]
    qids = [entry["qid"] for entry in selected]

    if (
        len(qids) != 495
        or len(set(qids)) != len(qids)
        or set(qids) != set(
            generation["configuration"]["selected_qids"]["test"]
        )
    ):
        raise ValueError(
            "Expected exactly the 495 previously "
            "selected CWQ test questions"
        )

    train_qids = set(
        generation["configuration"]["selected_qids"]["train"]
    )
    dev_qids = set(
        generation["configuration"]["selected_qids"]["dev"]
    )

    if set(qids) & (train_qids | dev_qids):
        raise ValueError(
            "Selected test QIDs overlap train/development"
        )

    audit = read_json(
        ROOT
        / "outputs/cwq_audit_v1"
        / "cwq_dataset_audit_summary_v1.json"
    )

    if (
        audit["signature"]
        != generation["configuration"]["audit_signature"]
    ):
        raise ValueError(
            "CWQ source audit differs from Notebook 12"
        )

    source_files = audit["configuration"]["source_files"]

    print("=" * 100)
    print(
        "NOTEBOOK 16: CWQ FROZEN TEST "
        "AND COST-MATCHED EVALUATION"
    )
    print("=" * 100)

    print(
        "GPU:", torch.cuda.get_device_name(0),
        "| Transformers:", transformers.__version__,
    )
    print(
        "Test questions:", len(qids),
        "| Settings:", json.dumps(settings),
    )
    print(
        f"Development mean SetStop actions: {dev_mean:.6f}; "
        f"fixed comparison cap: {matched_budget}"
    )
    print(
        "Inference only. No thresholds, models, or budgets "
        "are selected using test performance."
    )

    for name in [
        "dev_simple.json",
        "test_simple.json",
        "entities.txt",
        "relations.txt",
    ]:
        print("Verifying source:", name, flush=True)

        path = data / name
        expected = source_files[name]

        if (
            path.stat().st_size != expected["bytes"]
            or file_hash(path) != expected["sha256"]
        ):
            raise ValueError(
                f"Source file changed: {name}"
            )

    identity = dict(
        controller_adapter=file_hash(
            adapter_dir / "adapter_model.safetensors"
        ),
        controller_configuration=file_hash(
            adapter_dir / "controller_config.json"
        ),
        answer_model=file_hash(
            answer_dir / "model.safetensors"
        ),
        answer_configuration=file_hash(
            answer_dir / "answer_recognizer_config.json"
        ),
    )

    if identity != dev_protocol["models"]:
        raise ValueError(
            "Models differ from those evaluated "
            "in Notebook 15"
        )

    protocol = dict(
        code_version=CODE_VERSION,
        data_signature=generation["signature"],
        manifest_sha256=file_hash(manifest_path),
        models=identity,
        settings=settings,
        methods=METHODS,
        test_qids=qids,
        test_qid_sha256=fingerprint(sorted(qids)),
        development_protocol_signature=(
            dev_protocol["signature"]
        ),
        development_summary_sha256=file_hash(
            dev_output / "cwq_dev_online_summary_v1.json"
        ),
        matched_budget_selection=dict(
            split="development",
            mean_setstop_actions=dev_mean,
            rule=(
                "Nearest integer to Notebook 15 mean "
                "SetStop actions; round half upward."
            ),
            selected_budget=matched_budget,
            caveat=(
                "Approximate action matching based on "
                "development; edges are not matched."
            ),
        ),
        source_sha256={
            name: source_files[name]["sha256"]
            for name in [
                "dev_simple.json",
                "test_simple.json",
                "entities.txt",
                "relations.txt",
            ]
        },
        controller_batch=CONTROLLER_BATCH,
        answer_batch=ANSWER_BATCH,
        controller_max_length=(
            controller_config["maximum_sequence_length"]
        ),
        answer_max_length=(
            answer_config["maximum_sequence_length"]
        ),
        gain_class_weights=GAIN_WEIGHTS.tolist(),
        seed=SEED,
        answer_accumulation=(
            "Maximum probability across encountered paths; "
            "monotonic accepted set."
        ),
        software_versions=versions,
        pytorch_version=torch.__version__,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        primary_comparator="model_fixed_b14_e2e",
        regression_questions=DEV_REGRESSION_QUESTIONS,
        test_used_for_selection=False,
    )

    signature = fingerprint(protocol)
    protocol_path = output / "cwq_test_protocol_v1.json"

    if (
        protocol_path.exists()
        and read_json(protocol_path).get("signature")
        != signature
    ):
        raise ValueError(
            "An existing test protocol differs. "
            "Restore its settings/software before resuming."
        )

    save_json(
        protocol_path,
        dict(signature=signature, **protocol),
    )

    cache = (
        output / "question_cache" / signature[:16]
    )
    cache.mkdir(parents=True, exist_ok=True)

    def cache_path(qid):
        filename = (
            hashlib.sha256(qid.encode()).hexdigest()[:24]
            + ".json"
        )
        return cache / filename

    completed = {}

    for qid in qids:
        path = cache_path(qid)

        if not path.exists():
            continue

        cached = read_json(path)

        if (
            cached.get("signature") != signature
            or cached.get("qid") != qid
        ):
            raise ValueError(
                f"Cached result identity differs: {qid}"
            )

        payload = {
            key: value
            for key, value in cached.items()
            if key != "payload_sha256"
        }

        if (
            cached.get("payload_sha256")
            != fingerprint(payload)
        ):
            raise ValueError(
                f"Cached result content hash failed: {qid}"
            )

        if (
            len(cached["records"]) != len(METHODS)
            or {
                row["policy"]
                for row in cached["records"]
            } != set(METHODS)
            or any(
                row["qid"] != qid
                for row in cached["records"]
            )
            or not cached["shared_prefix_check"]
        ):
            raise ValueError(
                f"Cached question/policy records invalid: {qid}"
            )

        completed[qid] = cached

    print(
        "Completed test questions reused:",
        len(completed),
        flush=True,
    )

    started = time.time()

    regression_path = (
        output / "cwq_notebook15_regression_check_v1.json"
    )

    if len(completed) < len(qids):
        with open(
            data / "entities.txt", encoding="utf-8"
        ) as file:
            entity_to_id = {
                line.rstrip("\r\n"): index
                for index, line in enumerate(file)
            }

        with open(
            data / "relations.txt", encoding="utf-8"
        ) as file:
            relations = [
                line.rstrip("\r\n")
                for line in file
            ]

        relation_to_id = {
            name: index
            for index, name in enumerate(relations)
        }
        relation_tokens = [
            tokens(name) for name in relations
        ]

        quantization = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            llm_int8_skip_modules=["score"],
        )

        tokenizer_controller = AutoTokenizer.from_pretrained(
            str(adapter_dir),
            local_files_only=True,
        )
        tokenizer_controller.padding_side = "right"
        tokenizer_controller.truncation_side = "right"

        base = AutoModelForSequenceClassification.from_pretrained(
            controller_config["base_model"],
            revision=controller_config["base_revision"],
            num_labels=4,
            id2label={
                0: "gain_0",
                1: "gain_1",
                2: "gain_2",
                3: "gain_3_plus",
            },
            label2id={
                "gain_0": 0,
                "gain_1": 1,
                "gain_2": 2,
                "gain_3_plus": 3,
            },
            problem_type="single_label_classification",
            quantization_config=quantization,
            dtype=torch.float16,
            device_map={"": 0},
            token=False,
            cache_dir="/content/setstop_cwq_hf_cache_v1",
            attn_implementation="sdpa",
        )

        base.config.pad_token_id = (
            tokenizer_controller.pad_token_id
        )
        base.config.use_cache = False

        # Preserve the preparation used in Notebook 15.
        base = prepare_model_for_kbit_training(
            base,
            use_gradient_checkpointing=False,
        )

        with safe_open(
            str(adapter_dir / "adapter_model.safetensors"),
            framework="pt",
            device="cpu",
        ) as file:
            head_weights = []

            for key in file.keys():
                if "score" not in key or not key.endswith("weight"):
                    continue

                tensor = file.get_tensor(key)

                if tensor.ndim == 2 and tensor.shape[0] == 4:
                    head_weights.append(tensor)

        if len(head_weights) != 1:
            raise ValueError(
                "Saved adapter must contain exactly "
                "one trained classification head"
            )

        controller_model = PeftModel.from_pretrained(
            base,
            str(adapter_dir),
            is_trainable=False,
        )
        controller_model.eval()

        loaded_heads = [
            parameter
            for name, parameter in (
                controller_model.named_parameters()
            )
            if (
                "score" in name
                and "modules_to_save.default" in name
                and name.endswith("weight")
            )
        ]

        if (
            len(loaded_heads) != 1
            or not torch.equal(
                loaded_heads[0].detach().float().cpu(),
                head_weights[0].float(),
            )
        ):
            raise ValueError(
                "The trained controller classification "
                "head was not restored correctly"
            )

        tokenizer_answer = AutoTokenizer.from_pretrained(
            str(answer_dir),
            local_files_only=True,
        )
        tokenizer_answer.padding_side = "right"
        tokenizer_answer.truncation_side = "right"

        answer_model = (
            AutoModelForSequenceClassification.from_pretrained(
                str(answer_dir),
                local_files_only=True,
            )
        )
        answer_model.to("cuda").eval()

        for model in [
            controller_model,
            answer_model,
        ]:
            for parameter in model.parameters():
                parameter.requires_grad_(False)

        def make_predictor(
            model,
            tokenizer,
            max_length,
            batch_size,
            classes,
        ):
            # Memoization is scoped to one question.
            memo = {}

            def predict(texts):
                missing = list(
                    dict.fromkeys(
                        text
                        for text in texts
                        if text not in memo
                    )
                )

                order = sorted(
                    range(len(missing)),
                    key=lambda index: len(missing[index]),
                )

                with torch.inference_mode():
                    for start in range(
                        0, len(order), batch_size
                    ):
                        batch_texts = [
                            missing[index]
                            for index in order[
                                start:start + batch_size
                            ]
                        ]

                        encoded = tokenizer(
                            batch_texts,
                            padding=True,
                            truncation=True,
                            max_length=max_length,
                            pad_to_multiple_of=8,
                            return_tensors="pt",
                        )

                        encoded = {
                            key: value.to("cuda")
                            for key, value in encoded.items()
                        }

                        with torch.autocast(
                            device_type="cuda",
                            dtype=torch.float16,
                        ):
                            logits = model(
                                **encoded,
                                return_dict=True,
                            ).logits

                        values = (
                            torch.softmax(
                                logits.float(), dim=-1
                            )
                            .cpu()
                            .numpy()
                        )

                        if (
                            values.shape
                            != (len(batch_texts), classes)
                            or not np.isfinite(values).all()
                        ):
                            raise ValueError(
                                "Invalid model probabilities"
                            )

                        for text, value in zip(
                            batch_texts, values
                        ):
                            memo[text] = value.copy()

                values = np.asarray(
                    [memo[text] for text in texts],
                    dtype=np.float32,
                ).reshape(len(texts), classes)

                return (
                    values
                    if classes == 4
                    else values[:, 1]
                )

            return predict

        def predict_question(graph, gold):
            return evaluate_question(
                graph,
                gold,
                relations,
                relation_tokens,
                make_predictor(
                    controller_model,
                    tokenizer_controller,
                    controller_config["maximum_sequence_length"],
                    CONTROLLER_BATCH,
                    4,
                ),
                make_predictor(
                    answer_model,
                    tokenizer_answer,
                    answer_config["maximum_sequence_length"],
                    ANSWER_BATCH,
                    2,
                ),
                settings,
            )

        regression_entries = (
            manifest["splits"]["dev"][
                :DEV_REGRESSION_QUESTIONS
            ]
        )

        regression_qids = {
            entry["qid"]
            for entry in regression_entries
        }

        old_records = {}

        with open(
            dev_output / "cwq_dev_online_per_question_v1.jsonl",
            encoding="utf-8",
        ) as file:
            for line in file:
                row = json.loads(line)

                if row["qid"] in regression_qids:
                    old_records[
                        (row["qid"], row["policy"])
                    ] = row

        wanted_lines = {
            int(entry["line_number"]): entry
            for entry in regression_entries
        }

        checked = []

        print(
            "Checking three development questions "
            "against Notebook 15...",
            flush=True,
        )

        with open(
            data / "dev_simple.json",
            encoding="utf-8",
        ) as file:
            for line_number, line in enumerate(file, 1):
                entry = wanted_lines.get(line_number)

                if entry is None:
                    continue

                raw = json.loads(line)

                if str(raw["id"]) != entry["qid"]:
                    raise ValueError(
                        "Regression development question changed"
                    )

                graph, gold = parse_question(
                    raw,
                    entity_to_id,
                    relation_to_id,
                    relations,
                )

                result = predict_question(graph, gold)

                for row in result["records"]:
                    if row["policy"] not in [
                        "relation_fixed_e2e",
                        "model_fixed_e2e",
                        "setstop_e2e",
                    ]:
                        continue

                    previous = old_records[
                        (row["qid"], row["policy"])
                    ]

                    for key in [
                        "predicted_answers",
                        "gold_answers",
                        "actions",
                        "retrieved_edges",
                        "true_positive_count",
                        "false_positive_count",
                        "false_negative_count",
                        "stop_reason",
                    ]:
                        if row[key] != previous[key]:
                            raise ValueError(
                                "Notebook 15 reproduction failed: "
                                f"{row['qid']}, "
                                f"{row['policy']}, {key}"
                            )

                checked.append(graph["qid"])

                if len(checked) == len(regression_entries):
                    break

        if set(checked) != regression_qids:
            raise ValueError(
                "Development regression questions "
                "were not all checked"
            )

        save_json(
            regression_path,
            dict(
                signature=signature,
                passed=True,
                questions=checked,
                trained_controller_head_restored=True,
                comparison=(
                    "Exact predictions and counts "
                    "from Notebook 15"
                ),
            ),
        )

        print(
            "Notebook 15 regression: PASS. "
            "Starting frozen test evaluation.",
            flush=True,
        )

        by_line = {
            int(entry["line_number"]): entry
            for entry in selected
        }

        if len(by_line) != len(selected):
            raise ValueError(
                "Duplicate selected test line number"
            )

        with open(
            data / "test_simple.json",
            encoding="utf-8",
        ) as file:
            for line_number, line in enumerate(file, 1):
                entry = by_line.get(line_number)

                if (
                    entry is None
                    or entry["qid"] in completed
                ):
                    continue

                raw = json.loads(line)

                question_hash = hashlib.sha256(
                    " ".join(
                        raw["question"].lower().split()
                    ).encode()
                ).hexdigest()

                if (
                    str(raw["id"]) != entry["qid"]
                    or question_hash
                    != entry["question_sha256"]
                ):
                    raise ValueError(
                        "Selected test question changed"
                    )

                graph, gold = parse_question(
                    raw,
                    entity_to_id,
                    relation_to_id,
                    relations,
                )

                if len(gold) != entry["gold_count"]:
                    raise ValueError(
                        "Selected test gold count changed"
                    )

                result = predict_question(graph, gold)

                result["signature"] = signature
                result["payload_sha256"] = fingerprint(result)

                save_json(
                    cache_path(graph["qid"]),
                    result,
                )

                completed[graph["qid"]] = result

                if (
                    len(completed) % 10 == 0
                    or len(completed) == len(qids)
                ):
                    elapsed = (
                        time.time() - started
                    ) / 60

                    print(
                        f"Questions: {len(completed):,}/"
                        f"{len(qids):,} | "
                        f"this invocation {elapsed:.1f} min",
                        flush=True,
                    )

    if set(completed) != set(qids):
        raise ValueError(
            "Some selected test questions "
            "were not evaluated"
        )

    regression = read_json(regression_path)

    if (
        regression.get("signature") != signature
        or not regression.get("passed")
    ):
        raise ValueError(
            "A valid Notebook 15 regression report is required"
        )

    records = [
        row
        for qid in qids
        for row in completed[qid]["records"]
    ]

    summaries = summarize(records)
    paired = bootstrap_comparisons(records, qids)

    per_question_path = (
        output / "cwq_test_per_question_v1.jsonl"
    )
    temporary = per_question_path.with_name(
        per_question_path.name + ".partial"
    )

    with open(
        temporary, "w", encoding="utf-8"
    ) as file:
        for record in records:
            file.write(
                json.dumps(record, allow_nan=False)
                + "\n"
            )

    temporary.replace(per_question_path)

    diagnostics = {}

    for policy in METHODS:
        group = [
            row
            for row in records
            if row["policy"] == policy
        ]

        diagnostics[policy] = dict(
            error_categories=dict(
                collections.Counter(
                    row["error_category"]
                    for row in group
                )
            ),
            missed_by_search=sum(
                row["missed_by_search_count"]
                for row in group
            ),
            visited_but_not_predicted=sum(
                row["visited_gold_not_predicted_count"]
                for row in group
            ),
            scored_gold_rejected=sum(
                row["recognizer_false_negative_count"]
                for row in group
            ),
            visited_gold_unscored=sum(
                row["visited_gold_unscored_count"]
                for row in group
            ),
            premature_learned_stops=sum(
                row["premature_stop"]
                for row in group
            ),
            questions_with_frontier_truncation=sum(
                row["frontier_truncations"] > 0
                for row in group
            ),
        )

        if (
            diagnostics[policy]["missed_by_search"]
            + diagnostics[policy]["visited_but_not_predicted"]
            != summaries[policy]["false_negative_count"]
        ):
            raise ValueError(
                "Search/recognition error decomposition failed"
            )

    setstop = summaries["setstop_e2e"]
    differences = {}

    for policy in METHODS[:-1]:
        baseline = summaries[policy]

        differences[policy] = dict(
            macro_f1_difference=(
                setstop["macro_f1"]
                - baseline["macro_f1"]
            ),
            exact_match_difference=(
                setstop["exact_match"]
                - baseline["exact_match"]
            ),
            complete_recall_difference=(
                setstop["complete_recall"]
                - baseline["complete_recall"]
            ),
            mean_action_difference=(
                setstop["mean_actions"]
                - baseline["mean_actions"]
            ),
            mean_edge_difference=(
                setstop["mean_retrieved_edges"]
                - baseline["mean_retrieved_edges"]
            ),
            action_reduction_percent=(
                100
                * (
                    baseline["mean_actions"]
                    - setstop["mean_actions"]
                )
                / max(1e-15, baseline["mean_actions"])
            ),
            edge_reduction_percent=(
                100
                * (
                    baseline["mean_retrieved_edges"]
                    - setstop["mean_retrieved_edges"]
                )
                / max(
                    1e-15,
                    baseline["mean_retrieved_edges"],
                )
            ),
        )

    summary = dict(
        evaluation=(
            "Frozen end-to-end CWQ "
            "multi-answer test evaluation"
        ),
        signature=signature,
        data_signature=generation["signature"],
        test_questions=len(qids),
        test_qid_sha256=protocol["test_qid_sha256"],
        settings=settings,
        methods=summaries,
        setstop_minus_baselines=differences,
        primary_comparator="model_fixed_b14_e2e",
        matched_budget_selection=(
            protocol["matched_budget_selection"]
        ),
        shared_prefix_checks_passed=all(
            completed[qid]["shared_prefix_check"]
            for qid in qids
        ),
        notebook15_regression_passed=True,
        gold_used_for_policy=False,
        test_used_for_selection=False,
        test_used_for_evaluation=True,
        software_versions=versions,
        pytorch_version=torch.__version__,
        gpu=torch.cuda.get_device_name(0),
        completed_question_caches=len(completed),
        runtime_minutes_this_invocation=(
            time.time() - started
        ) / 60,
        notes=[
            "Selected 495-question multi-answer subset; "
            "not the official full-CWQ benchmark.",
            "Gold answers are used for subset definition "
            "and post-search evaluation, never as search inputs.",
            "Fixed cap 14 is chosen from development "
            "mean actions, not from test outcomes.",
            "Action matching is approximate; report "
            "actual test actions and edge counts.",
            "Edge exposure is a structural proxy; "
            "branch metadata uses the supplied graph.",
            "This shared-prefix experiment is not a "
            "per-policy latency or FLOPs benchmark.",
            "Confidence intervals cover question sampling, "
            "not variation across training seeds.",
        ],
    )

    summary_path = output / "cwq_test_summary_v1.json"
    paired_path = (
        output / "cwq_test_paired_comparisons_v1.json"
    )
    diagnostics_path = (
        output / "cwq_test_diagnostics_v1.json"
    )

    paired.update(
        signature=signature,
        data_signature=generation["signature"],
    )

    save_json(summary_path, summary)
    save_json(paired_path, paired)
    save_json(
        diagnostics_path,
        dict(
            signature=signature,
            methods=diagnostics,
        ),
    )

    table_path = output / "cwq_test_results_v1.csv"

    columns = [
        "method",
        "questions",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "micro_f1",
        "exact_match",
        "complete_recall",
        "mean_actions",
        "mean_retrieved_edges",
        "mean_fraction_of_input_graph_exposed",
        "visited_gold_recall",
        "empty_prediction_rate",
    ]

    with open(
        table_path,
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file, fieldnames=columns
        )
        writer.writeheader()

        writer.writerows(
            {
                key: summaries[policy][key]
                for key in columns
            }
            for policy in METHODS
        )

    inventory_path = (
        output / "artifact_inventory_v1.json"
    )

    report_paths = [
        summary_path,
        paired_path,
        diagnostics_path,
        per_question_path,
        table_path,
        protocol_path,
        regression_path,
    ]

    save_json(
        inventory_path,
        dict(
            signature=signature,
            data_signature=generation["signature"],
            files=[
                dict(
                    name=path.name,
                    path=str(path),
                    bytes=path.stat().st_size,
                    sha256=file_hash(path),
                )
                for path in report_paths
            ],
        ),
    )

    print("\n" + "=" * 115)
    print("CWQ FROZEN TEST RESULTS")
    print("=" * 115)

    print(
        f"{'Method':<30} "
        f"{'F1':>8} "
        f"{'Micro F1':>10} "
        f"{'Exact':>8} "
        f"{'Complete':>10} "
        f"{'Actions':>9} "
        f"{'Edges':>10}"
    )

    for policy in METHODS:
        result = summaries[policy]

        print(
            f"{result['method']:<30} "
            f"{result['macro_f1']:>8.4f} "
            f"{result['micro_f1']:>10.4f} "
            f"{result['exact_match']:>8.4f} "
            f"{result['complete_recall']:>10.4f} "
            f"{result['mean_actions']:>9.2f} "
            f"{result['mean_retrieved_edges']:>10.2f}"
        )

    print(
        "\nSetStop minus comparator: "
        "macro-F1 difference and paired 95% CI"
    )

    for policy in METHODS[:-1]:
        result = paired["comparisons"][policy]["macro_f1"]
        low, high = result["ci95"]

        print(
            f"{METHOD_NAMES[policy]:<30} "
            f"{result['difference']:+.4f} "
            f"[{low:+.4f}, {high:+.4f}]"
        )

    print(
        "\nThresholds remain frozen:",
        settings["controller_threshold"],
        settings["answer_threshold"],
    )

    print("\nSEND THESE FOUR FILES:")

    for path in [
        summary_path,
        per_question_path,
        paired_path,
        inventory_path,
    ]:
        print(path)

    print(
        "\nCompleted. Preserve these test results "
        "without retuning against them."
    )


if __name__ == "__main__":
    main()

run_environment = os.environ.copy()
run_environment.update({
    "USE_TF": "0",
    "USE_FLAX": "0",
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "TOKENIZERS_PARALLELISM": "false",
})

result = subprocess.run(
    [sys.executable, "-u", "-c", CWQ_NOTEBOOK_16],
    env=run_environment,
)

if result.returncode:
    raise RuntimeError(
        "Evaluation stopped. Read the traceback above. "
        "Completed question caches are preserved; rerun this "
        "same cell after resolving the reported error."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 16: CWQ FROZEN TEST AND COST-MATCHED EVALUATION
GPU: Tesla T4 | Transformers: 5.17.0
Test questions: 495 | Settings: {"max_hops": 4, "frontier_limit": 128, "action_cap": 24, "controller_threshold": 0.853, "answer_threshold": 0.96, "matched_budget": 14}
Development mean SetStop actions: 13.619231; fixed comparison cap: 14
Inference only. No thresholds, models, or budgets are selected using test performance.
Verifying source: dev_simple.json
Verifying source: test_simple.json
Verifying source: entities.txt
Verifying source: relations.txt
Completed test questions reused: 495

CWQ FROZEN TEST RESULTS
Method                               F1   Micro F1    Exact   Complete   Actions      Edges
Relation fixed, cap 14           0.2968     0.3824   0.1697     0.3758     14.00     127.58
Relation fixed, cap 24           0.3512     0.4116   0.1960     0.4505    